In [1]:
import sys
import os
HOME = os.environ['HOME']
BASECODE_PATH = os.path.join(HOME,'international-brain-lab/prior-localization/decoding')
if not BASECODE_PATH in sys.path:                                                                              
     sys.path.insert(0, BASECODE_PATH)

fb = open('slurm_decode_base.py','r')
filestr = fb.read()
fb.close()
exec(filestr)

fiteidstr = """
sessdf = dut.query_sessions(selection=SESS_CRITERION)
sessdf = sessdf.sort_values('subject').set_index(['subject', 'eid'])
eid = sys.argv[1]
ADD_TO_SAVING_PATH = sys.argv[2]
filenames = fit_eid(eid,sessdf)

print('saving to files:',filenames)
"""

SLURM_DIRECTORY = os.path.join(SCRATCH,'international-brain-lab/prior-localization/slurm/')
py_file = '%s%s.py' % (SLURM_DIRECTORY, 'slurm_decode_eid')
with open(py_file,'w') as fp:
    fp.write(filestr)
    fp.write(fiteidstr)
    
def slurm_submit_python_file(py_file, inputstr, 
                             label = '',
                             n_days = 2,
                             n_hours = 0,
                             n_gigs_ram = 4,
                             partition_name = 'normal',
                             SLURM_DIRECTORY = '~/slurm/'):
    if not (SLURM_DIRECTORY[-1] == '/'):
        SLURM_DIRECTORY = SLURM_DIRECTORY + '/'
    if not os.path.exists(SLURM_DIRECTORY):
        os.mkdir(SLURM_DIRECTORY)
        
    job_file = '%s%s_%s.job' % (SLURM_DIRECTORY,inputstr,label)
    with open(job_file,'w') as fj:
        fj.writelines("#!/bin/bash\n")
        fj.writelines("#SBATCH --job-name=%s%s.job\n" % (SLURM_DIRECTORY,inputstr))
        fj.writelines("#SBATCH --output=%s%s.out\n" % (SLURM_DIRECTORY,inputstr))
        fj.writelines("#SBATCH --error=%s%s.err\n" % (SLURM_DIRECTORY,inputstr))
        fj.writelines("#SBATCH --time=%d-%d\n" % (n_days,n_hours))
        fj.writelines("#SBATCH --mem=%dG\n" % n_gigs_ram)
        fj.writelines("#SBATCH --qos=%s\n" % partition_name)
    #         fj.writelines("#SBATCH --mail-type=ALL\n")
    #         fj.writelines("#SBATCH --mail-user=$USER@stanford.edu\n")
        fj.writelines("python %s %s %s\n" %(py_file,inputstr,label))

    os.system("sbatch %s" %job_file)
    return

def slurm_outs(label='', SLURM_DIRECTORY=SLURM_DIRECTORY):
    for file in os.listdir(SLURM_DIRECTORY):
        if len(file) >= 4 and len(file) >= len(label) and file[-4:] == '.out' and label in file:
            yield file
            
def get_decoding_output_files(label = '', 
                              SLURM_DIRECTORY = SLURM_DIRECTORY):
    decoding_output_files = []
    for file in slurm_outs(label='', SLURM_DIRECTORY=SLURM_DIRECTORY):
        with open(SLURM_DIRECTORY+file,'r') as fr:
            for line in fr:
                if line[:16] == 'saving to files:':
                    decoding_output_files.extend([line_part for line_part in line.split('\'') if ('.pkl' in line_part) and (label in line_part)])

    return decoding_output_files


ADD_TO_SAVING_PATH = 'alleidV1'

In [2]:
sessdf = dut.query_sessions(selection=SESS_CRITERION)
sessdf = sessdf.sort_values('subject').set_index(['subject', 'eid'])
sessdf_eids = sessdf.index.unique(level='eid')
#
# sessdf_eids=np.unique(sessdf_eids)[30:32]
# sessdf_eids= ['dfd8e7df-dc51-4589-b6ca-7baccfeb94b4', '034e726f-b35f-41e0-8d6c-a22cc32391fb',
#             '7939711b-8b4d-4251-b698-b97c1eaa846e', 'fa704052-147e-46f6-b190-a65b837e605e',
#             '46794e05-3f6a-4d35-afb3-9165091a5a74', '1538493d-226a-46f7-b428-59ce5f43f0f9',
#             'b52182e7-39f6-4914-9717-136db589706e', '89f0d6ff-69f4-45bc-b89e-72868abb042a',
#             '3ce452b3-57b4-40c9-885d-1b814036e936', '2d5f6d81-38c4-4bdc-ac3c-302ea4d5f46e',
#             '4b7fbad4-f6de-43b4-9b15-c7c7ef44db4b', 'd839491f-55d8-4cbe-a298-7839208ba12b',
#             'd2918f52-8280-43c0-924b-029b2317e62c', 'c99d53e6-c317-4c53-99ba-070b26673ac4',
#             'ecb5520d-1358-434c-95ec-93687ecd1396', '5386aba9-9b97-4557-abcd-abc2da66b863',
#             '4b00df29-3769-43be-bb40-128b1cba6d35', '3663d82b-f197-4e8b-b299-7b803a155b84',
#             '85dc2ebd-8aaf-46b0-9284-a197aee8b16f', '15f742e1-1043-45c9-9504-f1e8a53c1744']
for eid in sessdf_eids:
    slurm_submit_python_file(py_file, 
                             eid, 
                             label = ADD_TO_SAVING_PATH,
                             n_gigs_ram = 16,
                             SLURM_DIRECTORY = SLURM_DIRECTORY)
    

Submitted batch job 44779034
Submitted batch job 44779035
Submitted batch job 44779036
Submitted batch job 44779037
Submitted batch job 44779038
Submitted batch job 44779039
Submitted batch job 44779040
Submitted batch job 44779041
Submitted batch job 44779042
Submitted batch job 44779043
Submitted batch job 44779044
Submitted batch job 44779045
Submitted batch job 44779046
Submitted batch job 44779047
Submitted batch job 44779048
Submitted batch job 44779049
Submitted batch job 44779050
Submitted batch job 44779051
Submitted batch job 44779052
Submitted batch job 44779053
Submitted batch job 44779054
Submitted batch job 44779055
Submitted batch job 44779056
Submitted batch job 44779057
Submitted batch job 44779058
Submitted batch job 44779059
Submitted batch job 44779060
Submitted batch job 44779061
Submitted batch job 44779062
Submitted batch job 44779063
Submitted batch job 44779064
Submitted batch job 44779065
Submitted batch job 44779066
Submitted batch job 44779067
Submitted batc

Submitted batch job 44779337
Submitted batch job 44779338
Submitted batch job 44779339
Submitted batch job 44779340
Submitted batch job 44779341
Submitted batch job 44779342
Submitted batch job 44779343
Submitted batch job 44779344
Submitted batch job 44779345
Submitted batch job 44779346
Submitted batch job 44779347
Submitted batch job 44779348
Submitted batch job 44779349
Submitted batch job 44779350
Submitted batch job 44779351
Submitted batch job 44779352
Submitted batch job 44779353
Submitted batch job 44779354
Submitted batch job 44779355
Submitted batch job 44779356
Submitted batch job 44779357
Submitted batch job 44779358
Submitted batch job 44779359
Submitted batch job 44779360
Submitted batch job 44779361
Submitted batch job 44779362
Submitted batch job 44779363
Submitted batch job 44779364
Submitted batch job 44779365
Submitted batch job 44779366
Submitted batch job 44779367
Submitted batch job 44779368
Submitted batch job 44779369
Submitted batch job 44779370
Submitted batc

In [2]:
os.system("squeue -u $USER");
get_decoding_output_files(label = ADD_TO_SAVING_PATH,
                          SLURM_DIRECTORY = SLURM_DIRECTORY)

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
          44771546    normal interact  bensonb  R    5:58:54      1 sh02-01n56


['/scratch/users/bensonb/international-brain-lab/prior-localization/decoding/ibl_witten_17/d832d9f7-c96a-4f63-8921-516ba4a7b61f/probe01/2022-02-08_root_decode_prior_optimal_bayesian_Lasso_align_goCue_times_0_pseudosessions_timeWindow_-0_6_-0_2_alleidV1.pkl',
 '/scratch/users/bensonb/international-brain-lab/prior-localization/decoding/ibl_witten_32/7502ae93-7437-4bcd-9e14-d73b51193656/probe01/2022-02-08_LP_decode_prior_optimal_bayesian_Lasso_align_goCue_times_0_pseudosessions_timeWindow_-0_6_-0_2_alleidV1.pkl',
 '/scratch/users/bensonb/international-brain-lab/prior-localization/decoding/CSHL055/0cbeae00-e229-4b7d-bdcc-1b0569d7e0c3/probe00/2022-02-08_CA1_decode_prior_optimal_bayesian_Lasso_align_goCue_times_0_pseudosessions_timeWindow_-0_6_-0_2_alleidV1.pkl',
 '/scratch/users/bensonb/international-brain-lab/prior-localization/decoding/CSHL055/0cbeae00-e229-4b7d-bdcc-1b0569d7e0c3/probe00/2022-02-08_SSp-bfd_decode_prior_optimal_bayesian_Lasso_align_goCue_times_0_pseudosessions_timeWindow_-

In [16]:
finished = get_decoding_output_files(label = ADD_TO_SAVING_PATH,
                          SLURM_DIRECTORY = SLURM_DIRECTORY)
#%%
indexers = ['subject', 'eid', 'probe', 'region']
indexers_neurometric = ['low_slope', 'high_slope', 'low_range', 'high_range', 'shift']
resultslist = []
for fn in finished:
    fo = open(fn, 'rb')
    result = pickle.load(fo)
    fo.close()
    tmpdict = {**{x: result[x] for x in indexers},
               'fold': -1,
               'mask': ''.join([str(item) for item in list(result['fit']['mask'].values * 1)]),
               'Rsquared_test': result['fit']['Rsquared_test_full'],
               **{f'Rsquared_test_pseudo{i}': result['pseudosessions'][i]['Rsquared_test_full']
                  for i in range(N_PSEUDO)}}
    if result['fit']['full_neurometric'] is not None \
            and np.all([result['pseudosessions'][i]['full_neurometric'] is not None for i in range(N_PSEUDO)]):
        tmpdict = {**tmpdict,
                   **{idx_neuro: result['fit']['full_neurometric'][idx_neuro]
                      for idx_neuro in indexers_neurometric},
                   **{str(idx_neuro) + f'_pseudo{i}': result['pseudosessions'][i]['full_neurometric'][idx_neuro]
                      for i in range(N_PSEUDO) for idx_neuro in indexers_neurometric}}
    resultslist.append(tmpdict)
    for kfold in range(result['fit']['nFolds']):
        tmpdict = {**{x: result[x] for x in indexers},
                   'fold': kfold,
                   'Rsquared_test': result['fit']['Rsquareds_test'][kfold],
                   'Best_regulCoef': result['fit']['best_params'][kfold],
                   **{f'Rsquared_test_pseudo{i}': result['pseudosessions'][i]['Rsquareds_test'][kfold]
                      for i in range(N_PSEUDO)},
                   }
        if result['fit']['fold_neurometric'] is not None:
            tmpdict = {**tmpdict,
                       **{idx_neuro: result['fit']['fold_neurometric'][kfold][idx_neuro]
                          for idx_neuro in indexers_neurometric}}
        if np.all([result['pseudosessions'][i]['fold_neurometric'] is not None for i in range(N_PSEUDO)]):
            tmpdict = {**tmpdict,
                       **{str(idx_neuro) + f'_pseudo{i}': result['pseudosessions'][i][
                           'fold_neurometric'][kfold][idx_neuro]
                          for i in range(N_PSEUDO) for idx_neuro in indexers_neurometric}
                       }
        resultslist.append(tmpdict)
resultsdf = pd.DataFrame(resultslist).set_index(indexers)

start_tw, end_tw = TIME_WINDOW
fn = OUTPUT_PATH + '_'.join([DATE, 'decode', TARGET,
                             dut.modeldispatcher[MODEL] if TARGET in ['prior', 'prederr'] else 'task',
                             ESTIMATORSTR, 'align', ALIGN_TIME, str(N_PSEUDO), 'pseudosessions',
                             'timeWindow', str(start_tw).replace('.', '_'), str(end_tw).replace('.', '_')])
if ADD_TO_SAVING_PATH != '':
    fn = fn + '_' + ADD_TO_SAVING_PATH
fn = fn + '.parquet'

metadata_df = pd.Series({'filename': fn, **fit_metadata})
metadata_fn = '.'.join([fn.split('.')[0], 'metadata', 'pkl'])
resultsdf.to_parquet(fn)
metadata_df.to_pickle(metadata_fn)

# command to close the ongoing placeholder
# client.close(); cluster.close()

# If you want to get the errors per-failure in the run:
# """
# failures = [(i, x) for i, x in enumerate(filenames) if x.status == 'error']
# for i, failure in failures:
# print(i, failure.exception(), failure.key)
# print(len(failures))
# print(np.array(failures)[:,0])
# print(len([(i, x) for i, x in enumerate(filenames) if x.status == 'pending']))
# import traceback
# tb = failure.traceback()
# traceback.print_tb(tb)
# """
# You can also get the traceback from failure.traceback and print via `import traceback` and
# traceback.print_tb()